#Shortened set-up:

In [5]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection
!mkdir -p /content/data
!unzip -oq "/content/drive/MyDrive/MMA3001/Pork Rasher Error-Packaging-.v4i.coco.zip" -d /content/data
DATA_DIR = "/content/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/MMA3001 Project/Pork-Tray-Inspection


# XX


In [6]:
import json
coco = json.load(open(f"{DATA_DIR}/train/_annotations.coco.json"))
print(coco.keys())
print(coco["images"][0])
print(coco["annotations"][0])
print(coco["categories"])

dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])
{'id': 0, 'license': 1, 'file_name': 'frame_with_red_8214_jpg.rf.5acda82172bcf022bd7f017eb084474d.jpg', 'height': 540, 'width': 720, 'date_captured': '2026-08-11T05:59:42+00:00', 'extra': {'name': 'frame_with_red_8214.jpg'}}
{'id': 1, 'image_id': 0, 'category_id': 2, 'bbox': [562.22, 387.78, 35.56, 28.89], 'iscrowd': 0, 'area': 1027.328, 'segmentation': []}
[{'id': 0, 'name': 'packaging-error', 'supercategory': 'none'}, {'id': 1, 'name': 'loose-meat', 'supercategory': 'packaging-error'}, {'id': 2, 'name': 'packaging-error', 'supercategory': 'packaging-error'}, {'id': 3, 'name': 'twisted-meat', 'supercategory': 'packaging-error'}, {'id': 4, 'name': 'unsealed', 'supercategory': 'packaging-error'}, {'id': 5, 'name': 'wrinkle', 'supercategory': 'packaging-error'}]


# Counting Images, Annotations, and Classes


In [7]:
from collections import Counter
import pandas as pd

SPLITS = ["train", "valid", "test"]
summary = []        # one row of totals per split
class_counts = {}   # instances of each class, per split

for split in SPLITS:
    coco = json.load(open(f"{DATA_DIR}/{split}/_annotations.coco.json"))
    images, anns = coco["images"], coco["annotations"]
    # Map category id -> name, e.g. 4 -> "unsealed"
    cat_names = {c["id"]: c["name"] for c in coco["categories"]}

    # An image is "empty" (PASS) if no annotation points to its id
    annotated_ids = {a["image_id"] for a in anns}
    n_empty = sum(1 for im in images if im["id"] not in annotated_ids)

    # Augmented copies share the original frame name in extra.name
    n_sources = len({im["extra"]["name"] for im in images})

    summary.append({
        "split": split,
        "images": len(images),
        "annotations": len(anns),
        "images without annotations": n_empty,
        "unique source frames": n_sources,
    })
    class_counts[split] = Counter(cat_names[a["category_id"]] for a in anns)

display(pd.DataFrame(summary).set_index("split"))
display(pd.DataFrame(class_counts).fillna(0).astype(int))

,images,annotations,images without annotations,unique source frames
split,,,,
train,3349,2923,684,1085
valid,120,121,14,120
test,80,79,12,80


,train,valid,test
packaging-error,1712,34,23
unsealed,1055,81,49
wrinkle,18,0,0
loose-meat,66,3,3
twisted-meat,72,3,4


# Creating Defect Bounding Boxes and Pulling Images

In [11]:
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

def show(split, image_entry, annotations, cat_names):
    """Display one image with its bounding boxes and class labels drawn on."""
    img = Image.open(f"{DATA_DIR}/{split}/{image_entry['file_name']}").convert("RGB")
    draw = ImageDraw.Draw(img)
    for a in annotations:
        if a["image_id"] == image_entry["id"]:
            x, y, w, h = a["bbox"]
            draw.rectangle([x, y, x + w, y + h], outline=(0, 255, 0), width=3)
            draw.text((x + 3, y + 3), cat_names[a["category_id"]], fill=(255, 255, 0))
    plt.figure(figsize=(5, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(image_entry["file_name"])
    plt.show()

random.seed(0)  # same "random" picks every run, so the notes stay valid

# Reload train explicitly: after step 3's loop, `coco` holds the test split
split = "train"
coco = json.load(open(f"{DATA_DIR}/{split}/_annotations.coco.json"))
anns = coco["annotations"]
cat_names = {c["id"]: c["name"] for c in coco["categories"]}
images_by_id = {im["id"]: im for im in coco["images"]}

# 2 random examples of each class
for cat_id, name in cat_names.items():
    img_ids = sorted({a["image_id"] for a in anns if a["category_id"] == cat_id})
    if not img_ids:
        continue  # skips the unused placeholder category 0
    print(f"=== {name}: {len(img_ids)} images contain this class ===")
    for img_id in random.sample(img_ids, min(2, len(img_ids))):
        show(split, images_by_id[img_id], anns, cat_names)

# 3 random images with no annotations (PASS images)
annotated = {a["image_id"] for a in anns}
empty = [im for im in coco["images"] if im["id"] not in annotated]
print(f"=== no annotations: {len(empty)} images ===")
for im in random.sample(empty, 3):
    show(split, im, anns, cat_names)

Output hidden; open in https://colab.research.google.com to view.

# Documentation

In [13]:
%%writefile docs/1_exploring_data.md
# Phase 1: Data exploration

Notebook: `notebooks/01_explore.ipynb`

## Steps

1. **Setup.** Mounted Google Drive, moved into the repository folder and unzipped
   the dataset to Colab's local disk (`/content/data`) for faster image reading.
2. **Annotation format.** Inspected the image, annotation and category entries of
   `_annotations.coco.json`. Each `bbox` is `[x, y, width, height]` in pixels,
   measured from the top-left corner of the image with y increasing downwards.
   Annotations link to images through `image_id` and to classes through
   `category_id`. Category 0 is an unused Roboflow placeholder.
3. **Counts.** Counted images, annotations, images without annotations and unique
   source frames per split, and the instances of each defect class.
4. **Annotation checks.** Checked that each box's area equals width × height, that
   every box lies inside the 720 × 540 image, and that every annotation refers to
   an existing image and category.
5. **Visual inspection.** Drew the annotated boxes and class labels on random
   examples of each class and on images without annotations.

## Results

### Split sizes

| Split | Images | Annotations | Images without annotations | Unique source frames |
|---|---|---|---|---|
| train | 3349 | 2923 | 684 | 1085 |
| valid | 120 | 121 | 14 | 120 |
| test | 80 | 79 | 12 | 80 |

### Defect instances per class

| Class | Train | Valid | Test |
|---|---|---|---|
| packaging-error | 1712 | 34 | 23 |
| unsealed | 1055 | 81 | 49 |
| wrinkle | 18 | 0 | 0 |
| loose-meat | 66 | 3 | 3 |
| twisted-meat | 72 | 3 | 4 |

### Annotation checks

All three checks passed for all splits: no area mismatches, no boxes outside the
image and no references to missing images or categories.

## Findings

- **Augmentation is applied to training only.** Train contains about three
  exposure-adjusted copies of each source frame; valid and test contain one
  unmodified image per frame, which is appropriate for evaluation.
- **Fewer source images than stated.** The export contains about 1,285 unique
  source frames across all splits, compared with the approximately 1,955 original
  images described in the project information.
- **Rare classes cannot be evaluated.** Wrinkle does not appear in valid or test,
  and loose meat and twisted meat have only 3–4 instances each, so a single miss
  would change their measured recall by 25–33 %.
- **Most images contain a defect.** 80 % of training images, 88 % of validation
  images and 85 % of test images have at least one annotation. Plain accuracy is
  therefore misleading: labelling every image as defective would score 85 % on test.
- **Small evaluation sets.** Valid (120 images) and test (80 images) are small,
  and test has only 12 images without defects, so metrics will carry wide
  uncertainty.
- **Images are video frames.** File names are numbered frames
  (`frame_with_red_<N>.jpg`), so nearby frames may show the same physical tray in
  different splits.

## Observations from example images for each class

- **Unsealed:** large boxes covering a wide part of the tray, marking where the
  film lid has not sealed. The defect is super subtle.
- **Packaging-error:** small boxes mostly at the tray rim or seal edge, where the meat meets the edge of the tray.
- **Loose meat:** boxes around a single rasher lying out of line with the others,
  often at the end of the tray.
- **Twisted meat:** medium boxes around a rasher that is simply twisted.
- **Wrinkle:** small boxes near the tray edge.
- **Images without annotations:** include trays only partly in view at the frame
  edge, so PASS means "no visible defect in this image" rather than "a complete
  good tray".
- **General:** many trays are cut off at the image edge, trays appear at varying
  angles, and the dark mesh conveyor fills most of each frame. Defects often lie
  close to the image edge.


Writing docs/1_exploring_data.md


# Commit

In [15]:
!git add notebooks/1_exploring_data.ipynb. docs/1_exploring_data.md
!git commit -m "Explore dataset"
!git push https://{token}@github.com/{chrixxv1}/{Pork-Tray-Inspection}.git HEAD

fatal: pathspec 'notebooks/1_exploring_data.ipynb.' did not match any files
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/0_setup.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	docs/
	notebooks/1_exploring_data.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
remote: Repository not found.
fatal: repository 'https://github.com/{chrixxv1}/{Pork-Tray-Inspection}.git/' not found
